this notebook aims to compare the VLE of acetone methanol:
* experimental data
* using GCM calculated values
* using corrected property for acetone

In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# step1: define the compounds
mixture = {'acetone': {'CH3':1, 'CH3CO':1},
           'methanol': {'CH3OH':1},
           'DOH': {'DOH':1}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

In [ ]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)

print(pure_props)
# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']



# step4: perform the vle
# define number of data points
n_data = 100
# set composition
x1 = np.linspace(0, 1, n_data)  # Component 1: 0 to 1
x2 = 1 - x1                     # Component 2: 1 to 0 (complement)
x = np.column_stack([x1, x2])  # Combine into 2D array
# provide nitial guess for temperature
# Extract boiling points
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points) 
T_max = max(boiling_points)  
T0 = np.linspace(T_min, T_max, n_data)
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

gamma_gcm = np.empty_like(x)
T_gcm = np.empty_like(T0)
y_gcm = np.empty_like(x)
psat_gcm = np.empty_like(x)

for i in model.gamma.keys():
    gamma_gcm[i] = model.gamma[i].value

for i in model.T.keys():
    T_gcm[i] = model.T[i].value

for i in model.y.keys():
    y_gcm[i] = model.y[i].value

for i in model.psat.keys():
    psat_gcm[i] = model.psat[i].value

now we fix the properties

In [ ]:
# fix the properties of acetone
pure_props['acetone']['tc'] = 508.1
pure_props['acetone']['pc'] = 48.0
pure_props['acetone']['tb'] = 329.3

# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props, method='enthalpy_vap')
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']

# perform the VLE calculation
results, model = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

gamma_exp = np.empty_like(x)
T_exp = np.empty_like(T0)
y_exp = np.empty_like(x)
psat_exp = np.empty_like(x)

for i in model.gamma.keys():
    gamma_exp[i] = model.gamma[i].value

for i in model.T.keys():
    T_exp[i] = model.T[i].value

for i in model.y.keys():
    y_exp[i] = model.y[i].value

for i in model.psat.keys():
    psat_exp[i] = model.psat[i].value

now we load the experimental data

In [ ]:
# load the vle data
import pandas as pd
df = pd.read_excel('../data/vle_exp.xlsx')

In [ ]:
df

In [ ]:
# Publication-quality T-x-y diagram with subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

# ========== LEFT PLOT: Full comparison with GCM ==========
# Plot GCM calculated curves
ax1.plot(x[:, 0], T_gcm, color='#2ca02c', linewidth=2.5, linestyle='--', 
        label='UNIFAC (GC properties)')
ax1.plot(y_gcm[:, 0], T_gcm, color='#2ca02c', linewidth=2.5, linestyle='--')

# Plot corrected property curves
ax1.plot(x[:, 0], T_exp, color='#1f77b4', linewidth=2.5, 
        label='UNIFAC (exp. properties)')
ax1.plot(y_exp[:, 0], T_exp, color='#1f77b4', linewidth=2.5)

# Plot experimental data
ax1.scatter(df['x'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
           linewidths=2, marker='o', label='Experimental', zorder=5)
ax1.scatter(df['y'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
           linewidths=2, marker='o', zorder=5)

# Axis labels
ax1.set_xlabel('Acetone mole fraction, $x_{\\mathrm{acetone}}$, $y_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax1.set_ylabel('Temperature [K]', fontsize=13, fontweight='normal')

# Limits
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(300, 340)

# Ticks
ax1.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend
ax1.legend(loc='upper right', fontsize=10, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Grid
ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

# Subplot label - lower left
ax1.text(0.02, 0.05, '(a)', transform=ax1.transAxes, fontsize=14, 
         fontweight='bold', va='bottom', ha='left',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1))


# ========== RIGHT PLOT: Without GCM (zoomed) ==========
# Plot corrected property curves only
ax2.plot(x[:, 0], T_exp, color='#1f77b4', linewidth=2.5, 
        label='UNIFAC (exp. properties)')
ax2.plot(y_exp[:, 0], T_exp, color='#1f77b4', linewidth=2.5)

# Plot experimental data
ax2.scatter(df['x'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
           linewidths=2, marker='o', label='Experimental', zorder=5)
ax2.scatter(df['y'], df['T'], s=80, facecolors='none', edgecolors='#d62728',
           linewidths=2, marker='o', zorder=5)

# Axis labels
ax2.set_xlabel('Acetone mole fraction, $x_{\\mathrm{acetone}}$, $y_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax2.set_ylabel('Temperature [K]', fontsize=13, fontweight='normal')

# Zoomed limits for better resolution
ax2.set_xlim(-0.02, 1.02)
ax2.set_ylim(327, 338)

# Ticks
ax2.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend
ax2.legend(loc='upper right', fontsize=10, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Grid
ax2.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

# Subplot label - lower left
ax2.text(0.02, 0.05, '(b)', transform=ax2.transAxes, fontsize=14, 
         fontweight='bold', va='bottom', ha='left',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1))

# Tight layout
plt.tight_layout()

# Save in multiple formats for different uses
# PDF for LaTeX (vector format, perfect quality at any scale)
plt.savefig('acetone_methanol_txy_comparison.pdf', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none', format='pdf')
# PNG for presentations/web
plt.savefig('acetone_methanol_txy_comparison.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none', format='png')

print("Figures saved as:")
print("  - acetone_methanol_txy_comparison.pdf (use this for LaTeX)")
print("  - acetone_methanol_txy_comparison.png (for presentations/web)")

plt.show()

we make xy diagram

In [ ]:
# Publication-quality x-y diagram
fig, ax = plt.subplots(figsize=(6.5, 6.5), dpi=300)

# Plot diagonal reference line first (in background)
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.6, label='$y = x$', zorder=1)

# Plot model predictions
ax.plot(x[:, 0], y_gcm[:, 0], color='#2ca02c', linewidth=2.5,
        linestyle='--', label='UNIFAC (GC properties)', zorder=3)
ax.plot(x[:, 0], y_exp[:, 0], color='#1f77b4', linewidth=2.5,
        label='UNIFAC (exp. properties)', zorder=4)

# Plot experimental data with hollow markers
ax.scatter(df['x'], df['y'], s=80, facecolors='none', edgecolors='#d62728',
           linewidths=2, marker='o', label='Experimental', zorder=5)

# Axis labels with proper formatting
ax.set_xlabel('Liquid mole fraction, $x_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')
ax.set_ylabel('Vapor mole fraction, $y_{\\mathrm{acetone}}$ [-]',
              fontsize=13, fontweight='normal')

# Set equal aspect ratio and limits
ax.set_aspect('equal')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Improve tick labels
ax.tick_params(axis='both', which='major', labelsize=11, direction='in',
               length=6, width=1.5, top=True, right=True)

# Legend with better positioning
ax.legend(loc='upper left', fontsize=10.5, frameon=True, fancybox=False,
          edgecolor='black', framealpha=1)

# Subtle grid
ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3, color='gray')

# Tight layout for better spacing
plt.tight_layout()

# Save in multiple formats for different uses
# PDF for LaTeX (vector format, perfect quality at any scale)
plt.savefig('acetone_methanol_xy_diagram.pdf', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none', format='pdf')
# PNG for presentations/web
plt.savefig('acetone_methanol_xy_diagram.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none', format='png')

print("Figures saved as:")
print("  - acetone_methanol_xy_diagram.pdf (use this for LaTeX)")
print("  - acetone_methanol_xy_diagram.png (for presentations/web)")

plt.show()

In [ ]:
from src.mixture.unifac.pyomo_azeo import pyo_azeo
T0_az = np.array((T_min+T_max)/2)
x0_az = np.array([[0.5, 0.5]])
results_az, model_az = pyo_azeo(x0_az, T0_az, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=False)

In [ ]:
# retrieving the solutions
gamma_az = np.empty_like(x0_az)
T_az = np.empty_like(T0_az)
y_az = np.empty_like(x0_az)
x_az = np.empty_like(x0_az)
psat_az = np.empty_like(x0_az)


for i in model_az.gamma.keys():
    gamma_az[i] = model_az.gamma[i].value

for i in model_az.T.keys():
    T_az = model_az.T[i].value

for i in model_az.y.keys():
    y_az[i] = model_az.y[i].value

for i in model_az.x.keys():
    x_az[i] = model_az.x[i].value

for i in model_az.psat.keys():
    psat_az[i] = model_az.psat[i].value
print('Azeotrope found at T=', T_az, 'K')
print('Liquid composition:', x_az)
print('Vapor composition:', y_az)